## Сравнение полного OOF с XGBoost

1. **Что делаем?** Показываем TabM, сохранённый XGBoost baseline и дельту одновременно.
2. **Зачем?** Итог должен быть проверяемым без ручного сопоставления чисел.
3. **Как это отвечает на вопрос?** Сравниваются шесть заранее определённых OOF-метрик.
4. **Что остаётся неизменным?** Метрики, порог 0.5 и baseline Этапа 1.

In [ ]:
def show_v4_oof_comparison():
    if 'om' not in globals(): return
    keys=('Gini','ROC-AUC','PR-AUC','Precision','Recall','F1'); rows_html=''.join(f'<tr><td>{key}</td><td>{float(XGB_OOF[key]):.5f}</td><td>{"—" if om is None else f"{om[key]:.5f}"}</td><td>{"—" if od is None else f"{od[key]:+.5f}"}</td></tr>' for key in keys); display(HTML(f'<h3>Полный OOF: TabM и XGBoost</h3><table><tr><th>Метрика</th><th>XGBoost</th><th>TabM</th><th>Δ TabM − XGBoost</th></tr>{rows_html}</table>'))
get_ipython().events.register('post_run_cell', lambda result: show_v4_oof_comparison() if 'om' in globals() else None)


# Этап 6 V4 — полный 3-fold OOF-прогон TabM

V3 завершился на одном внешнем фолде и дал предварительно неудовлетворительный результат. Одного фолда недостаточно для вывода. V4 проверяет устойчивость результата на полном 3-fold OOF; изменяется только число выполняемых внешних фолдов. Конфигурация TabM, данные и protocol V3 сохраняются. Время измеряется, но не ограничивает выполнение.

## Контракт эксперимента

1. **Что делаем?** Фиксируем данные, признаки, конфигурацию и V4 paths.
2. **Зачем?** Полный OOF сравним с V3 только при идентичном контракте.
3. **Как это отвечает на вопрос?** Код проверяет данные и 47 разрешённых признаков до обучения.
4. **Что остаётся неизменным?** Data_final.xlsb, DefMark, INN, исключение Q_B1_norm/Q_B2_norm, split, TabM и отсутствие tuning.

In [ ]:
from __future__ import annotations
import hashlib,json,os,platform,random,time,traceback
from pathlib import Path
import numpy as np,pandas as pd,tabm,torch,torch.nn.functional as F
from IPython.display import HTML,display
from sklearn.metrics import average_precision_score,f1_score,precision_score,recall_score,roc_auc_score
from sklearn.model_selection import StratifiedKFold,train_test_split
from torch.utils.data import DataLoader,TensorDataset
def root():
    for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents):
        if (p/'pyproject.toml').exists(): return p
    raise FileNotFoundError('Не найден корень проекта.')
def file_hash(p):
    h=hashlib.sha256()
    with p.open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def index_hash(x): return hashlib.sha256(np.asarray(x,dtype=np.int64).tobytes()).hexdigest()
def seed(s): random.seed(s);np.random.seed(s);torch.manual_seed(s);torch.use_deterministic_algorithms(True)
ROOT=root();DATASET=ROOT/'data'/'raw'/'Data_final.xlsb';BASELINE_PATH=ROOT/'reports'/'generated'/'stage1_baseline_results_V2.json'
GENERATED_DIR=ROOT/'reports'/'generated';SUMMARY_DIR=ROOT/'reports'/'summary';RESULTS_PATH=GENERATED_DIR/'stage6_tabm_results_V4.json';SUMMARY_PATH=SUMMARY_DIR/'stage6_tabm_summary_V4.json';OOF_PATH=GENERATED_DIR/'stage6_tabm_oof_V4.npz'
DATA_HASH='fc742be66d238c529daba52ccc755f774f836b7d052ed062cdf0b345080e7930';WORK_HASH='80430ce6290d0982d3641621ba1ed62f6fb495e8d32f7d23d9fca00091aadb45';TARGET='DefMark';IDENTIFIER='INN';FORBIDDEN=['Q_B1_norm','Q_B2_norm'];SEED=42;FOLD_SEEDS={1:43,2:44,3:45};MAX_EPOCHS=100;PATIENCE=16
CFG={'arch_type':'tabm','k':8,'n_blocks':3,'d_block':512,'activation':'ReLU','dropout':0.1,'start_scaling_init':'random-signs','d_out':2,'lr':0.002,'weight_decay':0.0003,'betas':(0.9,0.999),'eps':1e-8,'gradient_clip_global_norm':1.0,'batch_size':256}
CFG.update({'num_embeddings':None,'input_dtype':'float32','optimizer':'AdamW','share_training_batches':True,'max_epochs':MAX_EPOCHS,'amp':False,'torch_compile':False,'scheduler':None,'warmup':None,'class_weights':None,'sampling':None})
baseline=json.loads(BASELINE_PATH.read_text(encoding='utf-8'));FEATURES=baseline['допустимые_признаки'];XGB=baseline['модели']['XGBoost'];XGB_OOF=XGB['итоговые_метрики'];XGB_FOLDS=XGB['метрики_фолдов']
if len(FEATURES)!=47 or any(x in FEATURES for x in FORBIDDEN): raise ValueError('Нарушен контракт 47 разрешённых признаков.')


## Обучение и наблюдаемость

1. **Что делаем?** Определяем TabM, early stopping, refit и единую Jupyter-панель.
2. **Зачем?** Полный прогон должен быть наблюдаемым без строк по батчам.
3. **Как это отвечает на вопрос?** Панель обновляется после каждой эпохи с фолдом, лучшей эпохой, ROC-AUC и ETA.
4. **Что остаётся неизменным?** Архитектура, оптимизатор, batch size и ранняя остановка совпадают с V3.

In [ ]:
CURRENT=0;DISPLAY=None
def since(t): return time.perf_counter()-t
def show(stage,e,total,fs,all0,best=None,best_e=None,avg=None,eta=None):
    global DISPLAY
    et='—' if e is None else f'{e}/{total}';pct='—' if e is None else f'{round(100*e/total)}%';auc='пока нет' if best is None or not np.isfinite(best) else f'{best:.5f}';av='появится после первой эпохи' if avg is None else f'{avg/60:.2f} мин';left='пока нельзя оценить' if eta is None else f'{max(0,eta)/60:.1f} мин'
    h=HTML(f'<div style="font-family:Arial;border:1px solid #bbb;padding:10px"><h3>Этап 6 V4</h3><b>Фолд:</b> {CURRENT}/3<br><b>Этап:</b> {stage}<br><b>Эпоха:</b> {et}; <b>Прогресс:</b> {pct}<br><b>Лучшая эпоха:</b> {best_e if best_e else "пока нет"}; <b>Лучший ROC-AUC:</b> {auc}<br><b>Время фолда:</b> {since(fs)/60:.1f} мин; <b>Общее время:</b> {since(all0)/60:.1f} мин<br><b>Среднее время эпохи:</b> {av}; <b>Осталось:</b> {left}</div>')
    if DISPLAY is None: DISPLAY=display(h,display_id=True)
    else: DISPLAY.update(h)
def net(n): return tabm.TabM.make(n_num_features=n,cat_cardinalities=None,arch_type=CFG['arch_type'],k=CFG['k'],n_blocks=CFG['n_blocks'],d_block=CFG['d_block'],activation=CFG['activation'],dropout=CFG['dropout'],start_scaling_init=CFG['start_scaling_init'],num_embeddings=None,d_out=CFG['d_out'])
def make_loader(x,y,s,shuffle): return DataLoader(TensorDataset(torch.from_numpy(x),torch.from_numpy(y.astype(np.int64))),batch_size=CFG['batch_size'],shuffle=shuffle,generator=torch.Generator(device='cpu').manual_seed(s),num_workers=0)
def train_epoch(m,l,o):
    m.train()
    for x,y in l:
        o.zero_grad(set_to_none=True);z=m(x.float());target=y[:,None].expand(-1,m.k).reshape(-1);F.cross_entropy(z.reshape(-1,2),target).backward();torch.nn.utils.clip_grad_norm_(m.parameters(),CFG['gradient_clip_global_norm']);o.step()
@torch.inference_mode()
def predict(m,x):
    m.eval();t=torch.from_numpy(x).float();return np.concatenate([torch.softmax(m(t[i:i+CFG['batch_size']]),dim=-1).mean(dim=1)[:,1].cpu().numpy() for i in range(0,len(t),CFG['batch_size'])])
def score(y,p):
    q=(p>=.5).astype(np.int64);a=float(roc_auc_score(y,p));return {'ROC-AUC':a,'Gini':2*a-1,'PR-AUC':float(average_precision_score(y,p)),'Precision':float(precision_score(y,q,zero_division=0)),'Recall':float(recall_score(y,q,zero_division=0)),'F1':float(f1_score(y,q,zero_division=0))}
def select(x,y,s,fs,all0):
    a,b=train_test_split(np.arange(len(y)),test_size=.1,stratify=y,random_state=s);seed(s);m=net(x.shape[1]);o=torch.optim.AdamW(m.parameters(),lr=CFG['lr'],weight_decay=CFG['weight_decay'],betas=CFG['betas'],eps=CFG['eps']);l=make_loader(x[a],y[a],s,True);be,ba,stale,times=0,float('-inf'),0,[]
    for e in range(1,MAX_EPOCHS+1):
        t=time.perf_counter();train_epoch(m,l,o);v=float(roc_auc_score(y[b],predict(m,x[b])));be,ba,stale=(e,v,0) if v>ba else (be,ba,stale+1);times.append(since(t));avg=float(np.mean(times));show('выбор числа эпох',e,MAX_EPOCHS,fs,all0,ba,be,avg,avg*(MAX_EPOCHS-e))
        if stale>=PATIENCE: break
    return be,ba,e
def refit(x,y,v,s,epochs,ba,fs,all0):
    seed(s);m=net(x.shape[1]);o=torch.optim.AdamW(m.parameters(),lr=CFG['lr'],weight_decay=CFG['weight_decay'],betas=CFG['betas'],eps=CFG['eps']);l=make_loader(x,y,s,True);times=[]
    for e in range(1,epochs+1):
        t=time.perf_counter();train_epoch(m,l,o);times.append(since(t));avg=float(np.mean(times));show('полное обучение фолда',e,epochs,fs,all0,ba,epochs,avg,avg*(epochs-e))
    show('прогноз',epochs,epochs,fs,all0,ba,epochs);return predict(m,v),e


## Полный controlled experiment

1. **Что делаем?** Выполняем все три external folds, объединяем OOF и сохраняем V4 artifacts.
2. **Зачем?** Полный OOF нужен для проверки устойчивости V3.
3. **Как это отвечает на вопрос?** Получаются metrics по фолдам, полный OOF и deltas к XGBoost.
4. **Что остаётся неизменным?** StratifiedKFold с seed 42, seeds 43/44/45, отсутствие final test, tuning, class weights, balancing и threshold optimization.

In [ ]:
all0=time.perf_counter();run_status='running';error=None;rows=[];y_work=None;oof_p=oof_f=None
try:
    CURRENT=0;fs=all0;show('проверка данных',None,None,fs,all0)
    if file_hash(DATASET)!=DATA_HASH: raise ValueError('Контрольная сумма данных не совпадает.')
    data=pd.read_excel(DATASET,engine='pyxlsb');allowed=[c for c in data.columns if c not in [TARGET,IDENTIFIER,*FORBIDDEN]]
    if allowed!=FEATURES: raise ValueError('Порядок разрешённых признаков не совпадает.')
    x=data.loc[:,FEATURES].to_numpy(dtype=np.float32);y=data[TARGET].to_numpy(dtype=np.int64)
    if not np.isfinite(x).all(): raise ValueError('Обнаружены пропуски или нечисловые значения.')
    idx=np.arange(len(data));working,holdout=train_test_split(idx,test_size=.2,stratify=y,random_state=SEED)
    if len(working)!=289614 or index_hash(working)!=WORK_HASH: raise ValueError('Рабочее разбиение не совпадает.')
    x_work,y_work=x[working],y[working];del data,x,y,idx,holdout;oof_p=np.full(len(y_work),np.nan,dtype=np.float32);oof_f=np.zeros(len(y_work),dtype=np.int8)
    for fold,(tr,va) in enumerate(StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED).split(x_work,y_work),1):
        CURRENT=fold;fs=time.perf_counter();s=FOLD_SEEDS[fold];show('выбор числа эпох: начало',0,MAX_EPOCHS,fs,all0)
        be,ba,se=select(x_work[tr],y_work[tr],s,fs,all0);p,re=refit(x_work[tr],y_work[tr],x_work[va],s,be,ba,fs,all0);fm=score(y_work[va],p);oof_p[va]=p;oof_f[va]=fold;base=XGB_FOLDS[fold-1];delta={k:fm[k]-float(base[k]) for k in fm};rows.append({'fold':fold,'seed':s,'best_epoch':be,'inner_best_roc_auc':ba,'selection_epochs':se,'refit_epochs':re,'runtime_seconds':since(fs),'metrics':fm,'delta_vs_xgboost':delta});show('метрики фолда: завершены',be,be,fs,all0,ba,be,eta=float(np.mean([r['runtime_seconds'] for r in rows]))*(3-fold))
    run_status='completed'
except KeyboardInterrupt: run_status='interrupted';error={'тип':'KeyboardInterrupt','сообщение':'Запуск прерван пользователем.'}
except Exception as exc: run_status='error';error={'тип':type(exc).__name__,'сообщение':str(exc),'трассировка':traceback.format_exc()}
runtime=since(all0);complete=run_status=='completed' and oof_p is not None and np.isfinite(oof_p).all();om=score(y_work,oof_p) if complete else None;base={k:float(XGB_OOF[k]) for k in ('ROC-AUC','Gini','PR-AUC','Precision','Recall','F1')};od=None if om is None else {k:om[k]-base[k] for k in base}
decision='Эксперимент не завершён: использовать промежуточное состояние.' if od is None else ('Облегчённая TabM не дала прироста к сильному GBDT baseline.' if od['Gini']<=-.01 and od['PR-AUC']<=-.005 else 'Результат требует review по полным OOF-метрикам и времени.')
meta={'experiment':'Stage 6','version':'V4','status':run_status,'dataset_sha256':DATA_HASH,'target':TARGET,'identifier':IDENTIFIER,'feature_names_in_order':FEATURES,'feature_identity_sha256':hashlib.sha256('\n'.join(FEATURES).encode()).hexdigest(),'working_index_sha256':WORK_HASH,'final_test_used':False,'tabm_config':CFG,'outer_cv':{'type':'StratifiedKFold','n_splits':3,'shuffle':True,'random_state':42},'fold_seeds':FOLD_SEEDS,'fold_metrics':rows,'oof_metrics':om,'oof_complete':complete,'runtime_seconds':runtime,'comparison_vs_xgboost':{'oof_delta':od,'fold_deltas':[r['delta_vs_xgboost'] for r in rows]},'versions':{'python':platform.python_version(),'torch':torch.__version__,'tabm':tabm.__version__,'numpy':np.__version__,'cpu_count':os.cpu_count()},'limitations':['Random CV не доказывает temporal stability.','3 folds не являются statistical significance claim.','Порог 0.5 — диагностический.','Final test не использован.'],'decision':decision,'error':error};summary={'experiment':'Stage 6','version':'V4','status':run_status,'oof_metrics':om,'delta_vs_xgboost':od,'runtime_seconds':runtime,'decision':decision}
meta['early_stopping']={'inner_train_fraction':0.90,'inner_validation_fraction':0.10,'patience':16,'selection_metric':'ROC-AUC','refit_epochs':'best_epoch'}
GENERATED_DIR.mkdir(parents=True,exist_ok=True);SUMMARY_DIR.mkdir(parents=True,exist_ok=True);RESULTS_PATH.write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8');SUMMARY_PATH.write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
if complete: np.savez_compressed(OOF_PATH,y_true=y_work,oof_probability=oof_p,fold=oof_f)
table=''.join(f'<tr><td>{r["fold"]}</td><td>{r["best_epoch"]}</td><td>{r["metrics"]["Gini"]:.5f}</td><td>{r["metrics"]["PR-AUC"]:.5f}</td><td>{r["runtime_seconds"]/60:.1f}</td></tr>' for r in rows);text='OOF-метрики не получены.' if om is None else '; '.join(f'{k}: {v:.5f}' for k,v in om.items());dtext='Сравнение недоступно.' if od is None else '; '.join(f'Δ {k}: {v:+.5f}' for k,v in od.items());display(HTML(f'<h2>ИТОГ ЭТАПА 6 V4</h2><b>Статус:</b> {run_status}<br><b>Общий runtime:</b> {runtime/60:.1f} мин<table><tr><th>Фолд</th><th>Лучшая эпоха</th><th>Gini</th><th>PR-AUC</th><th>Минуты</th></tr>{table}</table><h3>Полный OOF</h3>{text}<h3>TabM vs XGBoost</h3>{dtext}<h3>Вывод</h3>{decision}'))


## ФАКТЫ

Панель показывает результаты фолдов, полный OOF и сравнение с XGBoost только по фактически завершённому прогону.

## ИНТЕРПРЕТАЦИЯ

Вывод зависит от фактических OOF-дельт и не предзаполнен как победа или поражение TabM.

## ОГРАНИЧЕНИЯ

Random CV не доказывает temporal stability. Три фолда не являются statistical significance claim. Порог 0.5 диагностический. Final test не использован.

## СЛЕДУЮЩИЙ ШАГ

Использовать V4 artifacts для review и принимать дальнейшее решение по полным OOF-метрикам и времени.